# 🔄 Module 2.4: Image Transformations — Geometry and Spatial Operations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_02_Image_Processing_Basics/04_Image_Transformations/image_transformations.ipynb)

---

## 🎯 Learning Objectives
1. Affine transformations (rotation, scaling, translation, shear)
2. Perspective (projective) transformations
3. Interpolation methods (nearest, bilinear, bicubic)
4. Geometric transforms as RL actions (crop, rotate, zoom)

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless -q

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# CIFAR-10: 60,000 real photographs in 10 classes (auto-downloads ~170MB)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10 loaded: {len(cifar10)} real photos (32x32x3)")
print(f"   Classes: {cifar10.classes}")

# scikit-image real test images
sample_images = {
    'astronaut': data.astronaut(),
    'camera': data.camera(), 
    'coins': data.coins(),
    'horse': data.horse(),
}
print(f"✅ scikit-image: {len(sample_images)} real test images loaded")

## Deep Mathematical Derivation: Affine and Projective Transformations

### Step 1: Homogeneous Coordinates
Represent 2D point $(x, y)$ as $\tilde{\mathbf{p}} = [x, y, 1]^T$ in homogeneous coordinates. This allows ALL affine transforms as matrix multiplication.

### Step 2: Affine Transformation (General Form)
$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

**Proof that affine transforms preserve parallel lines:**
Two parallel lines: $\mathbf{l}_1 = \mathbf{p}_1 + t\mathbf{d}$ and $\mathbf{l}_2 = \mathbf{p}_2 + t\mathbf{d}$ (same direction $\mathbf{d}$).

After affine transform $\mathbf{A}$: $\mathbf{A}\mathbf{l}_1 = \mathbf{A}\mathbf{p}_1 + t\mathbf{A}\mathbf{d}$ and $\mathbf{A}\mathbf{l}_2 = \mathbf{A}\mathbf{p}_2 + t\mathbf{A}\mathbf{d}$.

Both have direction $\mathbf{A}\mathbf{d}$ → still parallel. $\blacksquare$

### Step 3: Decomposition of Affine Transforms

**Rotation by angle $\theta$:**
$$\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

**Proof of orthogonality:** $\mathbf{R}^T\mathbf{R} = \mathbf{I}$ since $\cos^2\theta + \sin^2\theta = 1$. $\blacksquare$

**Proof of composition:** $\mathbf{R}(\alpha)\mathbf{R}(\beta) = \mathbf{R}(\alpha + \beta)$

$$\cos\alpha\cos\beta - \sin\alpha\sin\beta = \cos(\alpha+\beta)$$
$$\sin\alpha\cos\beta + \cos\alpha\sin\beta = \sin(\alpha+\beta)$$
by the angle addition formulas. $\blacksquare$

**Scaling:** $\mathbf{S}(s_x, s_y) = \text{diag}(s_x, s_y, 1)$

**Shear:** $\mathbf{H}(h_x, h_y) = \begin{bmatrix} 1 & h_x & 0 \\ h_y & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$

**Translation:** $\mathbf{T}(t_x, t_y) = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}$

**Theorem:** Any affine transform decomposes as $\mathbf{A} = \mathbf{T} \cdot \mathbf{R} \cdot \mathbf{S} \cdot \mathbf{H}$

### Step 4: Bilinear Interpolation (Full Derivation)
After transforming, output pixel $(x', y')$ maps to non-integer input location $(x, y)$.

**Given:** Four surrounding pixels $Q_{11}, Q_{12}, Q_{21}, Q_{22}$ at integer positions.

**Step 4a:** Interpolate in $x$-direction:
$$R_1 = \frac{x_2 - x}{x_2 - x_1} Q_{11} + \frac{x - x_1}{x_2 - x_1} Q_{21}$$
$$R_2 = \frac{x_2 - x}{x_2 - x_1} Q_{12} + \frac{x - x_1}{x_2 - x_1} Q_{22}$$

**Step 4b:** Interpolate in $y$-direction:
$$P = \frac{y_2 - y}{y_2 - y_1} R_1 + \frac{y - y_1}{y_2 - y_1} R_2$$

**Proof of uniqueness:** This is the unique polynomial $p(x,y) = a_0 + a_1 x + a_2 y + a_3 xy$ that matches $Q_{11}, Q_{12}, Q_{21}, Q_{22}$ (4 unknowns, 4 equations). $\blacksquare$

### Step 5: Projective (Perspective) Transformation
$$\begin{bmatrix} wx' \\ wy' \\ w \end{bmatrix} = \begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

Then: $x' = \frac{h_{11}x + h_{12}y + h_{13}}{h_{31}x + h_{32}y + h_{33}}, \quad y' = \frac{h_{21}x + h_{22}y + h_{23}}{h_{31}x + h_{32}y + h_{33}}$

**Proof that 4 point correspondences determine $\mathbf{H}$:**
$\mathbf{H}$ has 8 DOF (9 entries, but scale is arbitrary). Each point correspondence gives 2 equations. Therefore $4 \times 2 = 8$ equations → unique solution. $\blacksquare$

### RL Connection: Geometric Transforms as RL Actions
An RL agent for data augmentation can select transforms:
$$\mathcal{A} = \{R(\theta_1), R(\theta_2), S(s_1), S(s_2), T(t_1, t_2), \text{flip}_h, \text{flip}_v\}$$

The agent learns which augmentations improve downstream model performance — this is the core idea behind AutoAugment and learned augmentation policies.

## Homogeneous Coordinates — Why 3×3 Matrices for 2D Transforms

### The Problem with Cartesian Coordinates

In standard Cartesian coordinates, translation is NOT a linear operation:
$$\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} x + t_x \\ y + t_y \end{bmatrix} \neq \mathbf{M} \begin{bmatrix} x \\ y \end{bmatrix}$$

No $2 \times 2$ matrix can represent translation. This means we cannot compose rotations and translations using matrix multiplication alone.

### Solution: Embed 2D into 3D Projective Space

**Definition:** The homogeneous coordinate representation of $(x, y) \in \mathbb{R}^2$ is any triple $(wx, wy, w)$ with $w \neq 0$. The canonical form is $(x, y, 1)$.

**Recovery:** Given homogeneous $(X, Y, W)$, the Cartesian point is $(X/W, Y/W)$.

### Derivation: Translation as Linear Map in Homogeneous Coordinates

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} = \begin{bmatrix} x + t_x \\ y + t_y \\ 1 \end{bmatrix}$$

Now translation IS a matrix multiplication. The "trick" is that the extra dimension provides the constant term.

### Formal Proof: Composition of Arbitrary Affine Transforms

**Theorem:** The composition of two affine transforms $T_1$ then $T_2$ equals the matrix product $\mathbf{M}_2 \mathbf{M}_1$.

**Proof:** Let $\tilde{\mathbf{p}} = [x, y, 1]^T$. After $T_1$: $\tilde{\mathbf{p}}' = \mathbf{M}_1 \tilde{\mathbf{p}}$. After $T_2$: $\tilde{\mathbf{p}}'' = \mathbf{M}_2 \tilde{\mathbf{p}}' = \mathbf{M}_2 (\mathbf{M}_1 \tilde{\mathbf{p}}) = (\mathbf{M}_2 \mathbf{M}_1) \tilde{\mathbf{p}}$.

Therefore $T_2 \circ T_1$ has matrix $\mathbf{M}_{2 \circ 1} = \mathbf{M}_2 \mathbf{M}_1$. $\blacksquare$

**Critical consequence:** Matrix multiplication is NOT commutative ($\mathbf{M}_2 \mathbf{M}_1 \neq \mathbf{M}_1 \mathbf{M}_2$ in general), so the **order of transforms matters**. Rotating then translating gives a different result than translating then rotating.

### Counting Degrees of Freedom

| Transform Type | Matrix Structure | DOF | Preserves |
|:--------------|:----------------|:----|:----------|
| Rigid (Euclidean) | $\begin{bmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1 \end{bmatrix}$ | 3 | Distances, angles |
| Similarity | $\begin{bmatrix} s\cos\theta & -s\sin\theta & t_x \\ s\sin\theta & s\cos\theta & t_y \\ 0 & 0 & 1 \end{bmatrix}$ | 4 | Angles, ratios |
| Affine | $\begin{bmatrix} a & b & t_x \\ c & d & t_y \\ 0 & 0 & 1 \end{bmatrix}$ | 6 | Parallelism |
| Projective | $\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}$ | 8 | Cross-ratio |

Each level adds more DOF but preserves fewer geometric properties — a fundamental hierarchy in geometry (Klein's Erlangen program).

## Rotation Matrix from Complex Multiplication

The rotation matrix can be derived elegantly from the algebra of complex numbers, revealing a deep connection between geometry and algebra.

### Step 1: Complex Number Representation of 2D Points

Represent the point $(x, y) \in \mathbb{R}^2$ as the complex number $z = x + iy \in \mathbb{C}$.

### Step 2: Rotation as Complex Multiplication

**Claim:** Multiplying $z$ by $e^{i\theta}$ rotates the point by angle $\theta$ counterclockwise.

**Proof using Euler's formula:**
$$z' = e^{i\theta} \cdot z = (\cos\theta + i\sin\theta)(x + iy)$$
$$= (x\cos\theta - y\sin\theta) + i(x\sin\theta + y\cos\theta)$$

Extracting real and imaginary parts:
$$x' = x\cos\theta - y\sin\theta, \quad y' = x\sin\theta + y\cos\theta$$

In matrix form:
$$\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} \quad \blacksquare$$

### Step 3: Proof That Rotations Form a Group

**Closure:** $\mathbf{R}(\alpha)\mathbf{R}(\beta) = \mathbf{R}(\alpha + \beta)$

*Proof:* In complex form, $e^{i\alpha} \cdot e^{i\beta} = e^{i(\alpha+\beta)}$. The matrix product:
$$\begin{bmatrix} \cos\alpha & -\sin\alpha \\ \sin\alpha & \cos\alpha \end{bmatrix} \begin{bmatrix} \cos\beta & -\sin\beta \\ \sin\beta & \cos\beta \end{bmatrix} = \begin{bmatrix} \cos(\alpha+\beta) & -\sin(\alpha+\beta) \\ \sin(\alpha+\beta) & \cos(\alpha+\beta) \end{bmatrix}$$

by the angle addition formulas. $\blacksquare$

**Identity:** $\mathbf{R}(0) = \mathbf{I}$ $\checkmark$

**Inverse:** $\mathbf{R}(\theta)^{-1} = \mathbf{R}(-\theta) = \mathbf{R}(\theta)^T$ (since $\cos(-\theta) = \cos\theta$, $\sin(-\theta) = -\sin\theta$)

This group is $SO(2)$ — the special orthogonal group in 2D, characterized by $\det(\mathbf{R}) = \cos^2\theta + \sin^2\theta = 1$.

### Step 4: Rotation About an Arbitrary Center

To rotate about point $(c_x, c_y)$ instead of the origin:

$$\mathbf{M} = \mathbf{T}(c_x, c_y) \cdot \mathbf{R}(\theta) \cdot \mathbf{T}(-c_x, -c_y)$$

$$= \begin{bmatrix} \cos\theta & -\sin\theta & c_x(1-\cos\theta) + c_y\sin\theta \\ \sin\theta & \cos\theta & c_y(1-\cos\theta) - c_x\sin\theta \\ 0 & 0 & 1 \end{bmatrix}$$

**Derivation:** Translate center to origin → rotate → translate back. The composition is computed by multiplying the three $3\times3$ homogeneous matrices. $\blacksquare$

### Step 5: Small Angle Approximation

For small $\theta$ (e.g., RL agent making fine adjustments), $\cos\theta \approx 1$, $\sin\theta \approx \theta$:

$$\mathbf{R}(\theta) \approx \begin{bmatrix} 1 & -\theta \\ \theta & 1 \end{bmatrix} = \mathbf{I} + \theta\begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$$

The matrix $\begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}$ is the **generator** of rotations — small rotations are approximately linear perturbations in this direction. This linearization is useful for gradient-based RL policy optimization.

## 1. Affine Transformations

An affine transformation maps point $(x, y)$ to $(x', y')$:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \begin{bmatrix} a & b & t_x \\ c & d & t_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

### Specific Transformations:

**Translation:** $T = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}$

**Rotation by θ:** $R = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}$

**Scaling:** $S = \begin{bmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{bmatrix}$

**Shear:** $H = \begin{bmatrix} 1 & h_x & 0 \\ h_y & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$

### Composition:
$$T_{total} = T_n \cdot T_{n-1} \cdots T_2 \cdot T_1$$

Order matters! (Matrix multiplication is not commutative)

## 2. Interpolation

When transforming, new pixel positions may not fall on integer grid.

**Nearest Neighbor:** $I'(x', y') = I(\text{round}(x), \text{round}(y))$

**Bilinear:** Weighted average of 4 nearest pixels:
$$I'(x', y') = (1-a)(1-b)I(x_0, y_0) + a(1-b)I(x_1, y_0) + (1-a)b \cdot I(x_0, y_1) + ab \cdot I(x_1, y_1)$$

Where $a = x' - x_0$, $b = y' - y_0$

## Proof That Affine Maps Preserve Parallel Lines — and What They Don't Preserve

Affine transformations have a characteristic geometric property: they preserve parallelism. This section provides a rigorous proof and explores the hierarchy of preserved properties.

### Step 1: Formal Proof of Parallelism Preservation

**Theorem:** If lines $\ell_1$ and $\ell_2$ are parallel, their images under any affine map $T(\mathbf{x}) = \mathbf{A}\mathbf{x} + \mathbf{t}$ are also parallel (or coincident).

**Proof:** Parameterize the lines as:
$$\ell_1 = \{\mathbf{p}_1 + t\mathbf{d} : t \in \mathbb{R}\}, \quad \ell_2 = \{\mathbf{p}_2 + t\mathbf{d} : t \in \mathbb{R}\}$$

(Same direction vector $\mathbf{d}$ since they're parallel.)

Apply $T$:
$$T(\ell_1) = \{\mathbf{A}\mathbf{p}_1 + t\mathbf{A}\mathbf{d} + \mathbf{t} : t \in \mathbb{R}\} = \{\mathbf{p}_1' + t\mathbf{d}' : t \in \mathbb{R}\}$$
$$T(\ell_2) = \{\mathbf{A}\mathbf{p}_2 + t\mathbf{A}\mathbf{d} + \mathbf{t} : t \in \mathbb{R}\} = \{\mathbf{p}_2' + t\mathbf{d}' : t \in \mathbb{R}\}$$

where $\mathbf{d}' = \mathbf{A}\mathbf{d}$. Both image lines have the same direction $\mathbf{d}'$, so they're parallel. $\blacksquare$

### Step 2: What Affine Maps DO NOT Preserve

1. **Distances:** $\|T(\mathbf{x}_1) - T(\mathbf{x}_2)\| = \|\mathbf{A}(\mathbf{x}_1 - \mathbf{x}_2)\| \neq \|\mathbf{x}_1 - \mathbf{x}_2\|$ in general (unless $\mathbf{A}$ is orthogonal)

2. **Angles:** A shear transform maps a right angle to a non-right angle

3. **Areas (in general):** Area transforms by factor $|\det(\mathbf{A})|$

**Proof of area transformation:**
$$\text{Area}(T(S)) = |\det(\mathbf{A})| \cdot \text{Area}(S)$$
by the change of variables formula in integration. $\blacksquare$

### Step 3: Ratios of Areas on Parallel Lines ARE Preserved

**Theorem:** If points $A, B, C$ are collinear with $B$ between $A$ and $C$, then the ratio $\frac{|AB|}{|AC|}$ is preserved by any affine transform.

**Proof:** $B = A + \frac{|AB|}{|AC|}(C - A) = (1-\lambda)A + \lambda C$ where $\lambda = |AB|/|AC|$.

$$T(B) = \mathbf{A}[(1-\lambda)A + \lambda C] + \mathbf{t} = (1-\lambda)T(A) + \lambda T(C)$$

So $T(B)$ divides the segment $T(A)T(C)$ in the same ratio $\lambda$. $\blacksquare$

### Step 4: The Fundamental Theorem of Affine Geometry

**Theorem:** Any bijection $\mathbb{R}^n \to \mathbb{R}^n$ that maps lines to lines is an affine map.

This remarkable result means: if a transformation preserves collinearity (maps every set of 3 collinear points to 3 collinear points), it MUST be of the form $T(\mathbf{x}) = \mathbf{A}\mathbf{x} + \mathbf{t}$. Affine maps are completely characterized by their line-preserving property.

### RL Consequence

When an RL agent applies affine transforms for data augmentation:
- Parallel edges in objects remain parallel → object structure preserved
- Relative positions within objects maintained → recognition not disrupted
- But angles and distances change → forces the agent to learn rotation/scale invariance

## Forward vs. Inverse Mapping — Why Image Transforms Use Backward Warping

### Step 1: Forward Mapping (Intuitive but Problematic)

For each input pixel $(x, y)$, compute the output position:
$$(x', y') = T(x, y)$$

**Problem 1 — Holes:** Multiple input pixels may map to the same output pixel, while some output pixels receive no input — creating "holes" in the output.

**Problem 2 — Splatting:** When $(x', y')$ is non-integer, the input pixel's value must be distributed among nearby output pixels. This "splatting" is complex and can produce artifacts.

### Step 2: Inverse Mapping (Standard Approach)

For each output pixel $(x', y')$, compute the corresponding input position:
$$(x, y) = T^{-1}(x', y')$$

Then interpolate $I(x, y)$ from the input image.

**Advantage 1 — No holes:** Every output pixel gets a value (since we iterate over output pixels).

**Advantage 2 — Standard interpolation:** The non-integer position $(x, y)$ falls in the input image, where interpolation is well-defined.

### Step 3: Proof That Inverse Mapping Requires $T^{-1}$

**Theorem:** For an affine transform $\mathbf{M}$, the inverse is:

$$\mathbf{M}^{-1} = \frac{1}{\det(\mathbf{M})} \text{adj}(\mathbf{M})$$

For a $3 \times 3$ homogeneous matrix with the affine constraint (last row = $[0, 0, 1]$):

$$\begin{bmatrix} a & b & t_x \\ c & d & t_y \\ 0 & 0 & 1 \end{bmatrix}^{-1} = \frac{1}{ad - bc}\begin{bmatrix} d & -b & bt_y - dt_x \\ -c & a & ct_x - at_y \\ 0 & 0 & ad - bc \end{bmatrix}$$

**The inverse exists iff $\det = ad - bc \neq 0$** — i.e., the transform is non-degenerate (doesn't collapse the image to a line or point). $\blacksquare$

### Step 4: Computational Advantage of Inverse Mapping

**Forward mapping:** Must handle holes + splatting → complex, image-dependent
**Inverse mapping:** Simple per-pixel loop + standard interpolation → GPU-friendly, parallelizable

```
for each (x', y') in output:
    (x, y) = M_inv @ (x', y', 1)
    output[x', y'] = interpolate(input, x, y)
```

This is why OpenCV's `warpAffine` and `warpPerspective` use inverse mapping by default.

### Step 5: RL Implication

For an RL agent that applies geometric transformations:
- The agent specifies a forward transform $T$ (e.g., "rotate 15°")
- The implementation automatically computes $T^{-1}$ for inverse mapping
- The agent never sees holes or artifacts from forward mapping
- Transform parameters can be continuous actions for policy gradient methods

In [ ]:
# Create sample image
size = 100
img = np.zeros((size, size, 3), dtype=np.uint8)
# Colorful checkerboard
for i in range(0, size, 20):
    for j in range(0, size, 20):
        color = [np.random.randint(100, 255) for _ in range(3)]
        img[i:i+20, j:j+20] = color
# Add text-like features
cv2.putText(img, 'RL', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255), 3)

# Apply transformations
h, w = img.shape[:2]
center = (w//2, h//2)

# Rotation
M_rot = cv2.getRotationMatrix2D(center, 30, 1.0)
rotated = cv2.warpAffine(img, M_rot, (w, h))

# Scale
M_scale = np.float32([[1.5, 0, -25], [0, 1.5, -25]])
scaled = cv2.warpAffine(img, M_scale, (w, h))

# Shear
M_shear = np.float32([[1, 0.3, 0], [0.1, 1, 0]])
sheared = cv2.warpAffine(img, M_shear, (w+30, h+10))

# Translation
M_trans = np.float32([[1, 0, 20], [0, 1, 15]])
translated = cv2.warpAffine(img, M_trans, (w, h))

# Perspective
pts1 = np.float32([[0,0],[w,0],[0,h],[w,h]])
pts2 = np.float32([[10,20],[w-10,10],[20,h-10],[w-20,h-20]])
M_persp = cv2.getPerspectiveTransform(pts1, pts2)
perspective = cv2.warpPerspective(img, M_persp, (w, h))

# Flip
flipped = cv2.flip(img, 1)  # Horizontal flip

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
transforms = [
    (img, 'Original'),
    (rotated, 'Rotation (30°)'),
    (scaled, 'Scale (1.5×)'),
    (sheared, 'Shear'),
    (translated, 'Translation (+20,+15)'),
    (perspective, 'Perspective'),
    (flipped, 'Horizontal Flip'),
    (img, 'RL: Agent chooses\nwhich transform!'),
]

for idx, (result, title) in enumerate(transforms):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB) if len(result.shape)==3 else result)
    axes[row, col].set_title(title, fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Geometric Transformations — Each is an RL Action!',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('transformations.png', dpi=150, bbox_inches='tight')
plt.show()

## Bilinear Interpolation — Full Derivation and Error Analysis

When geometric transforms map output pixels to non-integer input locations, interpolation reconstructs the missing values. Bilinear interpolation offers the best quality-to-cost tradeoff.

### Step 1: Problem Setup

After applying an inverse geometric transform, output pixel $(x', y')$ maps to a non-integer input location $(x, y)$ where $x = \lfloor x \rfloor + \alpha$ and $y = \lfloor y \rfloor + \beta$, with $0 \leq \alpha, \beta < 1$.

The four surrounding integer-position pixels are:
$$Q_{00} = I(\lfloor x \rfloor, \lfloor y \rfloor), \quad Q_{10} = I(\lfloor x \rfloor + 1, \lfloor y \rfloor)$$
$$Q_{01} = I(\lfloor x \rfloor, \lfloor y \rfloor + 1), \quad Q_{11} = I(\lfloor x \rfloor + 1, \lfloor y \rfloor + 1)$$

### Step 2: Derivation as Tensor Product of Linear Interpolants

**Step 2a:** Interpolate in $x$ along the bottom row:
$$R_0 = (1 - \alpha) Q_{00} + \alpha Q_{10}$$

**Step 2b:** Interpolate in $x$ along the top row:
$$R_1 = (1 - \alpha) Q_{01} + \alpha Q_{11}$$

**Step 2c:** Interpolate in $y$ between the two results:
$$P = (1 - \beta) R_0 + \beta R_1$$

Expanding:
$$P = (1-\alpha)(1-\beta)Q_{00} + \alpha(1-\beta)Q_{10} + (1-\alpha)\beta Q_{01} + \alpha\beta Q_{11}$$

### Step 3: Proof That This Is the Unique Bilinear Polynomial

**Theorem:** There exists a unique polynomial $p(x, y) = a_0 + a_1 x + a_2 y + a_3 xy$ passing through the four corner values.

**Proof:** The polynomial has 4 unknowns $(a_0, a_1, a_2, a_3)$, and we have 4 equations:
$$p(0,0) = Q_{00}, \quad p(1,0) = Q_{10}, \quad p(0,1) = Q_{01}, \quad p(1,1) = Q_{11}$$

The system matrix is:
$$\begin{bmatrix} 1 & 0 & 0 & 0 \\ 1 & 1 & 0 & 0 \\ 1 & 0 & 1 & 0 \\ 1 & 1 & 1 & 1 \end{bmatrix} \begin{bmatrix} a_0 \\ a_1 \\ a_2 \\ a_3 \end{bmatrix} = \begin{bmatrix} Q_{00} \\ Q_{10} \\ Q_{01} \\ Q_{11} \end{bmatrix}$$

$\det = 1 \neq 0$, so the solution is unique. $\blacksquare$

### Step 4: Error Bounds

**Theorem:** For an image $I \in C^2$ (twice continuously differentiable), the bilinear interpolation error is:

$$|I(x,y) - P(x,y)| \leq \frac{1}{4}\left(\max|I_{xx}| + \max|I_{yy}| + \max|I_{xy}|\right) \cdot h^2$$

where $h$ is the pixel spacing.

**Key insight:** The error is $O(h^2)$, compared to nearest-neighbor which is $O(h)$. Bicubic interpolation achieves $O(h^4)$ but costs $16 \times$ more computation (using a 4×4 neighborhood).

### Step 5: Comparison of Interpolation Methods

| Method | Neighborhood | Accuracy | Cost | Best For |
|:-------|:-------------|:---------|:-----|:---------|
| Nearest | $1 \times 1$ | $O(h)$ | 1 mult | Speed-critical RL |
| Bilinear | $2 \times 2$ | $O(h^2)$ | 4 mult | General purpose |
| Bicubic | $4 \times 4$ | $O(h^4)$ | 16 mult | High quality |
| Lanczos | $6 \times 6$ | $O(h^6)$ | 36 mult | Publication quality |

For RL agents that must process many frames per second, bilinear interpolation is the typical choice — good enough quality at affordable computational cost.

## Projective Geometry — Cross-Ratio Invariance

Projective transformations are the most general linear image transforms. They model perspective effects (objects appearing smaller with distance). The cross-ratio is the unique projective invariant.

### Step 1: Projective Transformation (Homography)

$$\begin{bmatrix} wx' \\ wy' \\ w \end{bmatrix} = \begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

With dehomogenization: $x' = \frac{h_{11}x + h_{12}y + h_{13}}{h_{31}x + h_{32}y + h_{33}}$, $y' = \frac{h_{21}x + h_{22}y + h_{23}}{h_{31}x + h_{32}y + h_{33}}$

**Key difference from affine:** The denominator is NOT constant (depends on $x, y$). This creates the perspective effect where parallel lines converge to a vanishing point.

### Step 2: Degrees of Freedom

$\mathbf{H}$ has 9 entries but is defined up to scale ($\mathbf{H}$ and $\alpha\mathbf{H}$ give the same transformation), so DOF $= 9 - 1 = 8$.

**Proof that 4 point correspondences determine H:**

Each correspondence $\tilde{\mathbf{p}}_i' = \mathbf{H}\tilde{\mathbf{p}}_i$ gives 2 independent equations (the 3rd is dependent due to the homogeneous scale). With 4 correspondences: $4 \times 2 = 8$ equations for 8 unknowns → unique solution (generically). $\blacksquare$

### Step 3: The Cross-Ratio — Definition

For four collinear points $A, B, C, D$ with coordinates $a, b, c, d$ along the line:

$$\text{CR}(A, B; C, D) = \frac{(c - a)(d - b)}{(c - b)(d - a)}$$

### Step 4: Proof of Projective Invariance

**Theorem:** The cross-ratio is invariant under projective transformations.

**Proof:** A 1D projective transform is a Möbius transformation: $f(x) = \frac{ax + b}{cx + d}$

Compute the cross-ratio of the transformed points $f(a), f(b), f(c), f(d)$:

$$f(c) - f(a) = \frac{ac + b}{cc + d} - \frac{aa + b}{ca + d} = \frac{(ac+b)(ca+d) - (aa+b)(cc+d)}{(cc+d)(ca+d)}$$

Numerator: $(ac+b)(ca+d) - (aa+b)(cc+d) = (ad - bc)(c - a)$

Therefore: $f(c) - f(a) = \frac{(ad-bc)(c-a)}{(cc+d)(ca+d)}$

The cross-ratio:
$$\text{CR}' = \frac{[f(c)-f(a)][f(d)-f(b)]}{[f(c)-f(b)][f(d)-f(a)]}$$

$$= \frac{(ad-bc)^2(c-a)(d-b) \cdot \cancel{(cb+d)(ca+d)(cd+d)(cb+d)}^{-1} \cdot \cancel{\ldots}}{(ad-bc)^2(c-b)(d-a) \cdot \cancel{\ldots}} = \frac{(c-a)(d-b)}{(c-b)(d-a)} = \text{CR} \quad \blacksquare$$

### Step 5: Practical Applications

1. **Camera calibration:** The cross-ratio of known collinear points is preserved under perspective projection, allowing camera parameter estimation
2. **Vanishing point detection:** Parallel lines in 3D meet at vanishing points in the image — their cross-ratio reveals 3D structure
3. **Document rectification:** An RL agent performing document scanning can use cross-ratio preservation to verify correct perspective correction

### Step 6: Hierarchy of Geometric Transforms

$$\text{Euclidean} \subset \text{Similarity} \subset \text{Affine} \subset \text{Projective}$$

Each level adds more freedom but preserves fewer properties:
- Euclidean preserves: distances, angles, areas
- Similarity preserves: angles, ratios of distances
- Affine preserves: parallelism, ratios of areas
- Projective preserves: cross-ratio, collinearity (only)

## Affine Transform Estimation — Least Squares from Point Correspondences

Given a set of corresponding point pairs, how do we estimate the best-fit affine transform? This is a fundamental problem in image registration and augmentation.

### Step 1: Setting Up the Linear System

An affine transform maps $(x_i, y_i) \to (x_i', y_i')$:

$$\begin{bmatrix} x_i' \\ y_i' \end{bmatrix} = \begin{bmatrix} a & b \\ c & d \end{bmatrix} \begin{bmatrix} x_i \\ y_i \end{bmatrix} + \begin{bmatrix} t_x \\ t_y \end{bmatrix}$$

For $n$ point correspondences, stack into a system:

$$\begin{bmatrix} x_1 & y_1 & 0 & 0 & 1 & 0 \\ 0 & 0 & x_1 & y_1 & 0 & 1 \\ x_2 & y_2 & 0 & 0 & 1 & 0 \\ 0 & 0 & x_2 & y_2 & 0 & 1 \\ \vdots & & & & & \vdots \end{bmatrix} \begin{bmatrix} a \\ b \\ c \\ d \\ t_x \\ t_y \end{bmatrix} = \begin{bmatrix} x_1' \\ y_1' \\ x_2' \\ y_2' \\ \vdots \end{bmatrix}$$

Compact form: $\mathbf{A}\mathbf{p} = \mathbf{b}$ where $\mathbf{A} \in \mathbb{R}^{2n \times 6}$, $\mathbf{p} \in \mathbb{R}^6$.

### Step 2: Minimum Point Correspondences

The system has 6 unknowns, so we need at least 3 non-collinear point correspondences ($2 \times 3 = 6$ equations). With exactly 3 points, the solution is unique.

### Step 3: Least Squares Solution (Overdetermined System)

With $n > 3$ correspondences, the system is overdetermined. The least squares solution minimizes:

$$\mathbf{p}^* = \arg\min_{\mathbf{p}} \|\mathbf{A}\mathbf{p} - \mathbf{b}\|^2$$

**Normal equations:** $\mathbf{A}^T\mathbf{A}\mathbf{p}^* = \mathbf{A}^T\mathbf{b}$

**Proof:** Take gradient and set to zero:
$$\nabla_\mathbf{p} \|\mathbf{A}\mathbf{p} - \mathbf{b}\|^2 = 2\mathbf{A}^T(\mathbf{A}\mathbf{p} - \mathbf{b}) = 0 \implies \mathbf{p}^* = (\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T\mathbf{b} \quad \blacksquare$$

### Step 4: RANSAC for Robust Estimation

In practice, correspondences contain outliers (wrong matches). RANSAC provides robustness:

1. Randomly sample 3 correspondences
2. Estimate affine transform from these 3 points (exact solution)
3. Count inliers: points where $\|\mathbf{A}\mathbf{p}_i' - \mathbf{b}_i'\| < \epsilon$
4. Repeat $k$ times, keep transform with most inliers
5. Refine using least squares on all inliers

**Number of iterations** for probability $p$ of finding an outlier-free sample with inlier ratio $w$:
$$k = \frac{\log(1 - p)}{\log(1 - w^3)}$$

For $w = 0.5$ (50% inliers), $p = 0.99$: $k = \frac{\log(0.01)}{\log(1 - 0.125)} = 35$ iterations.

### Step 5: RL for Adaptive Registration

An RL agent performing image registration can:
- **State:** Current alignment error map
- **Action:** Adjust transform parameters $(a, b, c, d, t_x, t_y)$
- **Reward:** Decrease in alignment error (e.g., mutual information or SSD)

Unlike fixed-pipeline registration, the RL agent learns to handle difficult cases (large deformations, low texture regions) through experience.

## Bicubic Interpolation and Lanczos Resampling — Higher-Order Methods

For high-quality image transformations, bilinear interpolation may not suffice. Here we derive the mathematics of higher-order interpolation schemes.

### Step 1: Cubic Convolution Interpolation

Bicubic interpolation uses a 4×4 neighborhood. The 1D cubic convolution kernel (Keys, 1981):

$$u(s) = \begin{cases} (a+2)|s|^3 - (a+3)|s|^2 + 1 & 0 \leq |s| < 1 \\ a|s|^3 - 5a|s|^2 + 8a|s| - 4a & 1 \leq |s| < 2 \\ 0 & |s| \geq 2 \end{cases}$$

where $a$ controls the sharpness. The standard choice $a = -0.5$ gives the unique cubic that matches the ideal sinc function up to third-order accuracy.

**Proof that $a = -0.5$ gives $O(h^3)$ accuracy:**

The interpolation error depends on the moments of $u(s)$. For $a = -0.5$:
$$\sum_k u(s - k) = 1 \quad \text{(partition of unity) } \checkmark$$
$$\sum_k k \cdot u(s - k) = s \quad \text{(reproduces linears) } \checkmark$$
$$\sum_k k^2 \cdot u(s - k) = s^2 \quad \text{(reproduces quadratics) } \checkmark$$

The Taylor expansion of the error begins at $O(h^3)$, giving one order better accuracy than bilinear ($O(h^2)$). $\blacksquare$

### Step 2: 2D Bicubic via Separability

The 2D bicubic kernel is the tensor product of 1D kernels:
$$U(x, y) = u(x) \cdot u(y)$$

The interpolated value uses a 4×4 grid of neighbors:
$$P(x, y) = \sum_{i=-1}^{2}\sum_{j=-1}^{2} Q_{ij} \cdot u(x - x_i) \cdot u(y - y_j)$$

**Computational cost:** 16 evaluations of $u$ + 16 multiplications per output pixel (vs. 4 for bilinear).

### Step 3: Lanczos Resampling

The Lanczos kernel uses a windowed sinc function:

$$L(x) = \begin{cases} \text{sinc}(x) \cdot \text{sinc}(x/a) & |x| < a \\ 0 & |x| \geq a \end{cases}$$

where $\text{sinc}(x) = \sin(\pi x)/(\pi x)$ and $a$ is the kernel radius (typically 2 or 3).

**Why sinc?** The ideal interpolation kernel (by the Shannon sampling theorem) is $\text{sinc}(x)$, which has a perfect low-pass frequency response: $\hat{h}(f) = \mathbf{1}_{|f| < 1/2}$. Since sinc extends infinitely, the Lanczos window truncates it smoothly.

**Lanczos-3 vs Bicubic:** Lanczos-3 ($a = 3$, using 6×6 neighborhoods) produces sharper results with fewer ringing artifacts than bicubic, at the cost of 36 multiplications per pixel.

### Step 4: Error Comparison

| Method | Support | Error Order | Ops/pixel | Ringing |
|:-------|:--------|:-----------|:----------|:--------|
| Nearest | 1×1 | $O(h)$ | 0 | None |
| Bilinear | 2×2 | $O(h^2)$ | 4 | None |
| Bicubic | 4×4 | $O(h^3)$ | 16 | Mild |
| Lanczos-3 | 6×6 | $O(h^4)$ | 36 | Minimal |

### Step 5: RL Tradeoff

For an RL agent processing video at 30fps:
- Training: use bilinear (speed matters more than quality)
- Deployment: use Lanczos-3 (quality matters, inference is one-shot)
- The agent can even learn to switch interpolation methods based on image content — smooth regions need only bilinear, while textured regions benefit from Lanczos

## Non-Rigid Transformations — Thin Plate Splines

While affine and projective transforms handle rigid or near-rigid mappings, many real-world deformations (facial expressions, medical image registration) require non-rigid transforms.

### Step 1: The Thin Plate Spline (TPS) Model

Given $N$ control point correspondences $(\mathbf{p}_i, \mathbf{q}_i)$, find a smooth mapping $f: \mathbb{R}^2 \to \mathbb{R}^2$ that:
1. Exactly interpolates: $f(\mathbf{p}_i) = \mathbf{q}_i$ for all $i$
2. Minimizes the bending energy (smoothness):

$$E_{\text{bend}} = \iint \left[\left(\frac{\partial^2 f}{\partial x^2}\right)^2 + 2\left(\frac{\partial^2 f}{\partial x \partial y}\right)^2 + \left(\frac{\partial^2 f}{\partial y^2}\right)^2\right] dx \, dy$$

### Step 2: Solution Form

**Theorem (Duchon, 1976):** The unique minimizer of $E_{\text{bend}}$ subject to interpolation constraints has the form:

$$f(\mathbf{x}) = \mathbf{a}^T\begin{bmatrix} 1 \\ x \\ y \end{bmatrix} + \sum_{i=1}^{N} w_i \cdot U(\|\mathbf{x} - \mathbf{p}_i\|)$$

where $U(r) = r^2 \log r$ is the TPS radial basis function (Green's function of the biharmonic equation $\nabla^4 f = 0$).

### Step 3: Computing the Weights

The system of equations for the $x$-component:

$$\begin{bmatrix} \mathbf{K} & \mathbf{P} \\ \mathbf{P}^T & \mathbf{0} \end{bmatrix} \begin{bmatrix} \mathbf{w} \\ \mathbf{a} \end{bmatrix} = \begin{bmatrix} \mathbf{q}_x \\ \mathbf{0} \end{bmatrix}$$

where:
- $K_{ij} = U(\|\mathbf{p}_i - \mathbf{p}_j\|)$ is the $N \times N$ kernel matrix
- $\mathbf{P} = [1, x_i, y_i]$ is the $N \times 3$ polynomial matrix
- $\mathbf{q}_x = [q_{1,x}, \ldots, q_{N,x}]^T$ are target x-coordinates
- $\mathbf{P}^T\mathbf{w} = \mathbf{0}$ ensures the affine part doesn't contribute to bending energy

### Step 4: Bending Energy in Terms of Weights

$$E_{\text{bend}} = \mathbf{w}^T \mathbf{K} \mathbf{w}$$

**Proof:** The bending energy of $\sum w_i U(\|\mathbf{x} - \mathbf{p}_i\|)$ can be shown (via Green's theorem) to equal $\sum_{i,j} w_i w_j U(\|\mathbf{p}_i - \mathbf{p}_j\|) = \mathbf{w}^T\mathbf{K}\mathbf{w}$. The affine part contributes zero bending energy. $\blacksquare$

### Step 5: Regularized TPS for Noisy Correspondences

When control points are noisy, we relax exact interpolation:

$$\min_f \sum_{i=1}^N \|f(\mathbf{p}_i) - \mathbf{q}_i\|^2 + \lambda E_{\text{bend}}$$

The solution replaces $\mathbf{K}$ with $\mathbf{K} + \lambda \mathbf{I}$ — a form of Tikhonov regularization. The parameter $\lambda$ controls the tradeoff between fidelity and smoothness.

### RL Application

An RL agent for non-rigid image registration learns to place control points and set correspondences:
- **State:** Source and target images overlaid
- **Actions:** Add/move control point, adjust $\lambda$
- **Reward:** Registration accuracy (mutual information or SSIM)
- The agent learns to focus control points where deformation is largest

## 3. RL for Automatic Image Alignment/Cropping

### MDP Formulation:
- **State:** Current image + crop region
- **Actions:** {rotate_cw_5°, rotate_ccw_5°, zoom_in, zoom_out, shift_left, shift_right, shift_up, shift_down}
- **Reward:** Composition quality score (rule of thirds, horizon alignment)
- **Goal:** Learn to auto-crop and align photos!

---
**Next:** Module 2.5 — Noise and Denoising